<a href="https://colab.research.google.com/github/NabilaAzzahra/rancangan-arsitektur-solusi-AI/blob/main/poc_rag_bank_nusantara_mandiri_lanjutan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lanjutan PoC: metrik kualitas dengan Ragas

Notebook ini **tidak membuat ulang jawaban**. Jawaban B0 dan R1 sudah ada di `hasil_mentah.csv` dan penilaian manusia sudah disetujui, jadi jawaban tidak boleh diganti. Notebook ini hanya menghitung metrik Ragas dari jawaban tersebut.

**Langkah**
1. Simpan API key di Colab: ikon kunci (Secrets) di kiri, nama `GOOGLE_API_KEY`, aktifkan *Notebook access*.
2. Jalankan sel satu per satu dari atas (**jangan Run all** sebelum sel 1 selesai).
3. Sel 1 memasang paket. Jika Colab meminta restart, restart lalu lanjut dari sel 2.
4. Sel 2 memeriksa kuota. Jika semua model `GAGAL 429`, **berhenti** dan coba lagi setelah kuota reset.
5. Sel 3 meminta unggahan `poc_lanjutan.zip`.
6. Sel 5 menghitung Ragas. Hasil tiap metrik disimpan ke file CSV.
7. Sel 6 membuat ringkasan dan mengunduh `hasil_poc.zip`.

In [1]:
!pip -q install "ragas>=0.2.10,<0.4" "langchain-community>=0.4,<0.4.2" langchain-google-genai langchain-huggingface sentence-transformers google-genai pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [2]:
import os, re, time, json, glob, zipfile, getpass, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---- Konfigurasi tetap (sama dengan eksperimen B0/R1) ----
MODEL = "gemini-3.5-flash-lite"      # model pembangkit jawaban pada eksperimen
K = 5; CHUNK_TOKENS = 500; OVERLAP = 0.10; SCORE_THRESHOLD = 0.30
DATA_DIR = "poc"
RUN_CONTEXT = False   # True = hitung juga context precision/recall (lebih lama dan lebih banyak kuota)
JUDGE_CANDIDATES = ["gemini-3.5-flash-lite"]   # sama dengan model pembangkit, sesuai Subbab 5.5

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    os.environ.setdefault("GOOGLE_API_KEY", getpass.getpass("GOOGLE_API_KEY: "))

# ---- Cek kuota: pilih model penilai pertama yang bisa dipakai ----
from google import genai
_c = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
JUDGE_MODEL = None
for m in JUDGE_CANDIDATES:
    t = time.time()
    try:
        r = _c.models.generate_content(model=m, contents="Balas dengan satu kata: halo")
        print(m, "OK", round(time.time() - t, 1), "dtk")
        JUDGE_MODEL = JUDGE_MODEL or m
    except Exception as e:
        print(m, "GAGAL", round(time.time() - t, 1), "dtk ->", str(e)[:250])
if JUDGE_MODEL is None:
    raise SystemExit("Tidak ada model yang bisa dipakai (kuota habis?). Berhenti di sini dan coba lagi setelah kuota reset.")
print("\nModel penilai Ragas:", JUDGE_MODEL, "| pembangkit jawaban:", MODEL)

gemini-3.5-flash-lite OK 0.7 dtk

Model penilai Ragas: gemini-3.5-flash-lite | pembangkit jawaban: gemini-3.5-flash-lite


In [3]:
# Unggah poc_lanjutan.zip (berisi folder poc/, hasil_mentah.csv, lembar_penilaian_terisi.csv)
if not (os.path.isdir(DATA_DIR) and os.path.exists("hasil_mentah.csv")):
    if not os.path.exists("poc_lanjutan.zip"):
        from google.colab import files
        files.upload()
    with zipfile.ZipFile("poc_lanjutan.zip") as z:
        z.extractall(".")
assert os.path.exists("hasil_mentah.csv") and os.path.isdir(DATA_DIR), "File belum lengkap: cek isi zip"
print("OK:", sorted(os.listdir(DATA_DIR)), "| hasil_mentah.csv ada")

OK: ['docs', 'pertanyaan_uji.csv'] | hasil_mentah.csv ada


In [4]:
from transformers import AutoTokenizer
try:
    _tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
except Exception:
    _tok = None
class _E: pass
embedder = _E(); embedder.tokenizer = _tok

def tok_len(text):
    try:
        return len(embedder.tokenizer.encode(text, add_special_tokens=False))
    except Exception:
        return int(len(text.split()) * 1.3)

def parse_doc(path):
    txt = open(path, encoding="utf-8").read()
    head, _, body = txt.partition("\n## ")
    meta = {}
    for line in head.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    sections = []
    for part in re.split(r"\n(?=## )", "## " + body):
        part = part.strip()
        if not part:
            continue
        title, _, text = part.partition("\n")
        sections.append((title[3:].strip(), text.strip()))
    return meta, sections

def split_long(text, max_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    if tok_len(text) <= max_tokens:
        return [text]
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        cur.append(s)
        if tok_len(" ".join(cur)) > max_tokens and len(cur) > 1:
            cur.pop()
            out.append(" ".join(cur))
            keep, n = [], 0
            for t in reversed(cur):
                n += tok_len(t)
                if n > max_tokens * overlap:
                    break
                keep.insert(0, t)
            cur = keep + [s]
    if cur:
        out.append(" ".join(cur))
    return out

CHUNKS = []
for path in sorted(glob.glob(os.path.join(DATA_DIR, "docs", "*.md"))):
    meta, secs = parse_doc(path)
    for title, text in secs:
        label = title.split(".")[0].strip()          # mis. "Pasal 2" atau "FAQ 1"
        for j, piece in enumerate(split_long(text)):
            cid = f"{meta['doc_id']}#{label}" + (f"-{j+1}" if j else "")
            header = f"[{meta['doc_id']} v{meta['versi']} | {meta['jenis']} | {label}] {title}"
            CHUNKS.append({"chunk_id": cid, "doc_id": meta["doc_id"], "versi": meta["versi"],
                           "jenis": meta["jenis"], "label": label, "text": header + "\n" + piece})
CHUNK_TEXT = {c["chunk_id"]: c["text"] for c in CHUNKS}
print(len(CHUNKS), "chunk;", "maks token per chunk =", max(tok_len(c["text"]) for c in CHUNKS))
Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
# ---- Perbaikan kompatibilitas Ragas dan langchain-community (jalankan SEBELUM sel Ragas) ----
import sys, types, importlib

def _stub_vertex():
    try:
        import langchain_community
    except Exception:
        return "langchain_community tidak terpasang"
    notes = []
    try:
        importlib.import_module("langchain_community.chat_models.vertexai")
    except Exception:
        m = types.ModuleType("langchain_community.chat_models.vertexai")
        m.ChatVertexAI = type("ChatVertexAI", (), {})
        sys.modules["langchain_community.chat_models.vertexai"] = m
        notes.append("stub chat_models.vertexai")
    try:
        import langchain_community.llms as _L
        if not hasattr(_L, "VertexAI"):
            _L.VertexAI = type("VertexAI", (), {})
            notes.append("stub llms.VertexAI")
    except Exception:
        pass
    return ", ".join(notes) or "tidak perlu stub"

print("Perbaikan:", _stub_vertex())
for k in [k for k in list(sys.modules) if k == "ragas" or k.startswith("ragas.")]:
    del sys.modules[k]                      # buang impor ragas yang setengah jadi

# Bila data hasil (variabel 'rows') hilang karena sesi dimulai ulang, bangun dari hasil_mentah.csv
if "rows" not in globals():
    _raw = pd.read_csv("hasil_mentah.csv", keep_default_na=False)
    Q = pd.read_csv(os.path.join(DATA_DIR, "pertanyaan_uji.csv"), keep_default_na=False)
    rows = []
    for _, q in Q.iterrows():
        x = {"id": q["id"], "tipe": q["tipe"], "aksi_diharapkan": q["aksi_diharapkan"]}
        for cfg in ("B0", "R1"):
            d = _raw[(_raw["id"] == q["id"]) & (_raw["konfigurasi"] == cfg)].iloc[0]
            x[cfg] = {"jawaban": d["jawaban"], "aksi": d["aksi"],
                      "konteks_teks": [CHUNK_TEXT[c] for c in str(d["konteks_ids"]).split("|") if c in CHUNK_TEXT]}
        rows.append(x)
    RAW = pd.read_csv("hasil_mentah.csv")
    RAW["token_total"] = RAW["token_in"] + RAW["token_out"]
    print("rows dibangun ulang dari hasil_mentah.csv:", len(rows))
else:
    print("rows sudah ada di memori:", len(rows))

print('Q:', len(Q), '| chunk:', len(CHUNK_TEXT), '| rows:', len(rows))

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

23 chunk; maks token per chunk = 89
Perbaikan: tidak perlu stub
rows dibangun ulang dari hasil_mentah.csv: 30
Q: 30 | chunk: 23 | rows: 30


In [5]:
import logging
logging.getLogger("ragas.executor").setLevel(logging.ERROR)   # galat TETAP ditampilkan

def gold_texts(cids):
    return [CHUNK_TEXT[c] for c in str(cids).split("|") if c in CHUNK_TEXT]

QI = Q.set_index("id")
from ragas import evaluate, EvaluationDataset, SingleTurnSample, RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

judge = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=JUDGE_MODEL))
lc_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="BAAI/bge-m3", encode_kwargs={"normalize_embeddings": True}))
# timeout longgar + retry terbatas: kuota gratis sering membuat panggilan menunggu lama
rc = RunConfig(max_workers=2, timeout=180, max_retries=3, max_wait=20)

def make_samples(cfg):
    ids, smp = [], []
    for x in rows:
        if x["tipe"] != "dalam_cakupan": continue
        res = x[cfg]
        if cfg == "R1" and res["aksi"] != "jawab": continue
        ctx = res["konteks_teks"] if cfg == "R1" else gold_texts(QI.loc[x["id"], "konteks_acuan"])
        smp.append(SingleTurnSample(user_input=QI.loc[x["id"], "pertanyaan"], response=res["jawaban"],
                                    retrieved_contexts=ctx, reference=QI.loc[x["id"], "jawaban_referensi"]))
        ids.append(x["id"])
    return ids, smp

def run_metric(cfg, metric, nama):
    # satu metrik per panggilan; hasil disimpan ke CSV supaya tidak hilang bila sesi terputus
    fn = f"ragas_{cfg}_{nama}.csv"
    if os.path.exists(fn):
        print(cfg, nama, "sudah ada, dilewati"); return pd.read_csv(fn)
    ids, smp = make_samples(cfg)
    t0 = time.time()
    out = evaluate(EvaluationDataset(samples=smp), metrics=[metric], run_config=rc, raise_exceptions=False, show_progress=True)
    df = out.to_pandas()
    col = [c for c in df.columns if c not in ("user_input", "retrieved_contexts", "response", "reference")][0]
    d = pd.DataFrame({"id": ids, col: df[col].values})
    d.to_csv(fn, index=False)
    print(f"{cfg} {nama}: rata-rata={d[col].mean():.3f}  terisi={d[col].notna().sum()}/{len(d)}  ({time.time()-t0:.0f} dtk)")
    return d

# strictness=1: Gemini 3.x menolak 'multiple candidates' (n>1) yang dipakai default Ragas (strictness=3)
M = {
  "faithfulness":       lambda: Faithfulness(llm=judge),
  "answer_relevancy":   lambda: ResponseRelevancy(llm=judge, embeddings=lc_emb, strictness=1),
  "context_precision":  lambda: LLMContextPrecisionWithReference(llm=judge),
  "context_recall":     lambda: LLMContextRecall(llm=judge),
}
PLAN = [("B0", "faithfulness"), ("R1", "faithfulness"),
        ("B0", "answer_relevancy"), ("R1", "answer_relevancy")]
if RUN_CONTEXT:
    PLAN += [("R1", "context_precision"), ("R1", "context_recall")]
RES = {}
for cfg, nama in PLAN:
    try:
        RES[(cfg, nama)] = run_metric(cfg, M[nama](), nama)
        d = RES[(cfg, nama)]
        if d.iloc[:, 1].notna().mean() < 0.5:
            print("BERHENTI: hasil", cfg, nama, "hampir kosong. Lihat pesan galat di atas (kuota 429?). File CSV metrik ini dihapus agar bisa diulang.")
            os.remove(f"ragas_{cfg}_{nama}.csv"); break
    except Exception as e:
        print("GAGAL", cfg, nama, repr(e)[:200]); break

def gabung(cfg):
    ds = [RES[k].set_index("id") for k in RES if k[0] == cfg]
    return pd.concat(ds, axis=1).reset_index() if ds else None
RAGAS_B0, RAGAS_R1 = gabung("B0"), gabung("R1")
for cfg, df in (("B0", RAGAS_B0), ("R1", RAGAS_R1)):
    if df is not None:
        df.to_csv(f"ragas_{cfg}.csv", index=False); print(cfg, df.drop(columns="id").mean().round(3).to_dict())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (8 outputs left out)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

B0 answer_relevancy: rata-rata=0.836  terisi=20/20  (67 dtk)


Evaluating:   0%|          | 0/19 [00:00<?, ?it/s]

R1 answer_relevancy: rata-rata=0.920  terisi=19/19  (87 dtk)
B0 {'faithfulness': 0.137, 'answer_relevancy': 0.836}
R1 {'faithfulness': 1.0, 'answer_relevancy': 0.92}


In [6]:
import glob, zipfile
RAW = pd.read_csv("hasil_mentah.csv", keep_default_na=False)
for c in ("skor_top1", "latensi_s", "token_in", "token_out"):
    RAW[c] = pd.to_numeric(RAW[c], errors="coerce")
RAW["token_total"] = RAW["token_in"] + RAW["token_out"]

def muat_ragas(cfg):
    ds = []
    for f in sorted(glob.glob(f"ragas_{cfg}_*.csv")):
        ds.append(pd.read_csv(f).set_index("id"))
    return pd.concat(ds, axis=1).reset_index() if ds else None
RAGAS_B0, RAGAS_R1 = muat_ragas("B0"), muat_ragas("R1")
for n, d in (("B0", RAGAS_B0), ("R1", RAGAS_R1)):
    if d is None: print("PERINGATAN: file ragas", n, "tidak ditemukan")
    else:
        d.to_csv(f"ragas_{n}.csv", index=False)
        print(n, "->", {k: round(float(v), 3) for k, v in d.drop(columns="id").mean().items()}, "| baris:", len(d))

def p95(x): return float(np.percentile(x, 95))
R1 = RAW[RAW.konfigurasi == "R1"]; B0 = RAW[RAW.konfigurasi == "B0"]
ins = R1[R1.tipe == "dalam_cakupan"]; sen = R1[R1.tipe == "sensitif"]; oos = R1[R1.tipe == "di_luar_cakupan"]
get = lambda df, col: (float(df[col].mean()) if df is not None and col in df.columns and df[col].notna().any() else np.nan)

S = {}
S["H1_faithfulness_B0"] = get(RAGAS_B0, "faithfulness")
S["H1_faithfulness_R1"] = get(RAGAS_R1, "faithfulness")
S["H2_faithfulness_R1>=0.85"] = S["H1_faithfulness_R1"]
S["H1_answer_relevancy_B0"] = get(RAGAS_B0, "answer_relevancy")
S["H1_answer_relevancy_R1"] = get(RAGAS_R1, "answer_relevancy")
S["R1_context_precision"] = get(RAGAS_R1, "llm_context_precision_with_reference")
S["R1_context_recall"] = get(RAGAS_R1, "context_recall")
S["H3_latensi_p95_R1_semua_s"] = p95(R1.latensi_s)
S["H3_latensi_p95_R1_dalam_cakupan_s"] = p95(ins.latensi_s)
S["H3_latensi_rata2_R1_s"] = float(R1.latensi_s.mean())
S["H3_latensi_p95_B0_s"] = p95(B0.latensi_s)
S["H4_eskalasi_sensitif_tepat_%"] = 100 * float((sen.aksi == "eskalasi").mean())
S["H4_penolakan_di_luar_cakupan_tepat_%"] = 100 * float((oos.aksi == "tolak").mean())
S["H4_dalam_cakupan_dijawab_tanpa_agen_%"] = 100 * float((ins.aksi == "jawab").mean())
S["H5_token_per_pertanyaan_R1_dijawab"] = float(ins[ins.aksi == "jawab"].token_total.mean())
S["H5_token_per_pertanyaan_R1_semua"] = float(R1.token_total.mean())
S["H5_token_per_pertanyaan_B0"] = float(B0.token_total.mean())
S["model_penilai_ragas"] = JUDGE_MODEL
SUM = pd.DataFrame({"metrik": list(S.keys()), "nilai": list(S.values())})
SUM.to_csv("ringkasan_poc.csv", index=False, encoding="utf-8-sig")
print(SUM.to_string(index=False))

# Distribusi skor retrieval (untuk pembahasan ambang)
print("\nSkor top-1 per tipe:\n", R1.groupby("tipe").skor_top1.describe()[["count", "min", "mean", "max"]])
Q = pd.read_csv(os.path.join("poc", "pertanyaan_uji.csv"), keep_default_na=False)
J = Q[["id", "tipe", "pertanyaan", "jawaban_referensi"]].copy()
rr = RAW.set_index(["id", "konfigurasi"])
J["jawaban_B0"] = [rr.loc[(i, "B0"), "jawaban"] for i in J.id]
J["jawaban_R1"] = [rr.loc[(i, "R1"), "jawaban"] for i in J.id]
J["aksi_R1"] = [rr.loc[(i, "R1"), "aksi"] for i in J.id]
J["nilai_B0"] = ""; J["nilai_R1"] = ""
if not os.path.exists("lembar_penilaian_terisi.csv"):
    J.to_csv("lembar_penilaian.csv", index=False, encoding="utf-8-sig")

with zipfile.ZipFile("hasil_poc.zip", "w") as z:
    for f in ("hasil_mentah.csv", "ringkasan_poc.csv", "ragas_B0.csv", "ragas_R1.csv"):
        if os.path.exists(f): z.write(f)
    for f in glob.glob("ragas_*_*.csv"): z.write(f)
try:
    from google.colab import files
    files.download("hasil_poc.zip")
except Exception:
    print("Unduh hasil_poc.zip dari panel File.")


B0 -> {'answer_relevancy': 0.836, 'faithfulness': 0.137} | baris: 20
R1 -> {'answer_relevancy': 0.92, 'faithfulness': 1.0} | baris: 19
                               metrik                 nilai
                   H1_faithfulness_B0              0.136806
                   H1_faithfulness_R1                   1.0
             H2_faithfulness_R1>=0.85                   1.0
               H1_answer_relevancy_B0               0.83632
               H1_answer_relevancy_R1              0.919539
                 R1_context_precision                   NaN
                    R1_context_recall                   NaN
            H3_latensi_p95_R1_semua_s             13.534655
    H3_latensi_p95_R1_dalam_cakupan_s             24.435513
                H3_latensi_rata2_R1_s              3.380648
                  H3_latensi_p95_B0_s             46.451307
         H4_eskalasi_sensitif_tepat_%                 100.0
 H4_penolakan_di_luar_cakupan_tepat_%                 100.0
H4_dalam_cakupan_dijawab_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>